# 09 - TransXion v2 Stress Test

Notebook này thực hiện stress test độc lập cho **transxion_v2** theo proposal khóa luận.
Dataset này không thay thế hai benchmark chính IEEE-CIS và BAF. Vai trò của nó là kiểm tra
giới hạn của lớp giải thích chọn lọc trong một bối cảnh AML bổ sung.

Giao thức bắt buộc: time split 60/20/20; predictor, calibrator và threshold được khóa trước
explanation; rule audit và policy selection dùng các phần validation riêng; test chỉ được
đánh giá sau khi policy đã khóa. Fixed coverage là 5%, 10%, 25% và 50%, trong đó 10% và 25%
là kết quả chính. Mọi phương pháp phải so với predictor-score-only ở cùng ngân sách.

Trong tên notebook, `v2` là revision 2 của paper được proposal dẫn chiếu; official repository không công bố một dataset-v2 tag riêng. Full run khóa vào `tx.csv`, repository revision và SHA-256 ghi trong config.

Kết quả quick/fixture chỉ kiểm tra code và không được dùng làm bằng chứng khóa luận.

In [ ]:
from pathlib import Path
from importlib import metadata as importlib_metadata
import importlib
import importlib.util
import json
import os
import subprocess
import sys

from packaging.requirements import Requirement
from packaging.utils import canonicalize_name

KAGGLE = Path("/kaggle").exists()
REPO_URL = "https://github.com/Tommyhuy1705/Explainable_NeuroSymbolic_Fraud_Detection.git"
BRANCH = "main"
KAGGLE_PROJECT_DIR = Path("/kaggle/working/Explainable_NeuroSymbolic_Fraud_Detection")

def sync_project() -> Path:
    if KAGGLE:
        if not KAGGLE_PROJECT_DIR.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(KAGGLE_PROJECT_DIR)],
                check=True,
            )
        elif not (KAGGLE_PROJECT_DIR / ".git").is_dir():
            raise RuntimeError(f"Expected a Git clone at {KAGGLE_PROJECT_DIR}")
        else:
            subprocess.run(
                ["git", "-C", str(KAGGLE_PROJECT_DIR), "pull", "--ff-only", "origin", BRANCH],
                check=True,
            )
        return KAGGLE_PROJECT_DIR.resolve()
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("Run this notebook inside the project or on Kaggle with Internet enabled")

PROJECT_ROOT = sync_project()
while str(PROJECT_ROOT) in sys.path:
    sys.path.remove(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

# A Kaggle kernel may survive a previous run. Never execute a stale src module.
for module_name in tuple(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]

QUICK_RUN = os.getenv("THESIS_STRESS_QUICK_RUN", "0") == "1"
USE_TEST_FIXTURE = os.getenv("THESIS_STRESS_TEST_FIXTURE", "0") == "1"
if USE_TEST_FIXTURE and not QUICK_RUN:
    raise ValueError("THESIS_STRESS_TEST_FIXTURE=1 requires THESIS_STRESS_QUICK_RUN=1")
STRICT_RUNTIME_REQUIREMENTS = not (QUICK_RUN and USE_TEST_FIXTURE)

OUTPUT_BASE = (
    Path("/kaggle/working/thesis_stress_outputs")
    if KAGGLE else PROJECT_ROOT / "results/stress/notebooks"
)
DATA_ROOTS = [Path("/kaggle/input")] if KAGGLE else [PROJECT_ROOT / "data/raw"]

RUNTIME_IMPORTS = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scikit-learn": "sklearn",
    "scipy": "scipy",
    "pyyaml": "yaml",
    "joblib": "joblib",
    "torch": "torch",
    "xgboost": "xgboost",
    "lightgbm": "lightgbm",
}

declared_requirements = {}
for raw_line in (PROJECT_ROOT / "requirements.txt").read_text(encoding="utf-8").splitlines():
    requirement_text = raw_line.split("#", maxsplit=1)[0].strip()
    if not requirement_text:
        continue
    requirement = Requirement(requirement_text)
    normalized_name = canonicalize_name(requirement.name)
    if normalized_name in RUNTIME_IMPORTS:
        declared_requirements[normalized_name] = requirement
missing_declarations = sorted(set(RUNTIME_IMPORTS).difference(declared_requirements))
if missing_declarations:
    raise RuntimeError(
        "requirements.txt has no declared range for runtime packages: "
        + ", ".join(missing_declarations)
    )

def runtime_package_status(package_name):
    requirement = declared_requirements[package_name]
    import_name = RUNTIME_IMPORTS[package_name]
    if importlib.util.find_spec(import_name) is None:
        return "missing", None, requirement
    try:
        installed_version = importlib_metadata.version(requirement.name)
    except importlib_metadata.PackageNotFoundError:
        installed_version = None
    # Vendor-managed Torch images can expose a fully usable import while the
    # distribution metadata has no Version field. Torch is never repaired in
    # this notebook, so reading its runtime version cannot create a stale
    # import after pip installation.
    if not installed_version and package_name == "torch":
        torch_module = importlib.import_module(import_name)
        installed_version = str(getattr(torch_module, "__version__", "")).strip() or None
    if not installed_version:
        return "metadata_missing", None, requirement
    if installed_version not in requirement.specifier:
        return "out_of_range", installed_version, requirement
    return "ok", installed_version, requirement

runtime_status = {
    package_name: runtime_package_status(package_name)
    for package_name in RUNTIME_IMPORTS
}
torch_status, torch_version, torch_requirement = runtime_status["torch"]
if torch_status != "ok" and STRICT_RUNTIME_REQUIREMENTS:
    observed = torch_version if torch_version is not None else torch_status
    raise ImportError(
        f"PyTorch environment is incompatible: observed {observed!r}, required "
        f"{torch_requirement}. Select a Kaggle image whose preinstalled Torch/CUDA wheel "
        "satisfies this range; this notebook will not install, upgrade, downgrade, or replace Torch."
    )

repair_requirements = [
    str(requirement)
    for package_name, (status, _, requirement) in runtime_status.items()
    if package_name != "torch" and status != "ok"
]
if repair_requirements and KAGGLE and STRICT_RUNTIME_REQUIREMENTS:
    # --no-deps ensures this repair cannot replace Kaggle's preinstalled
    # Torch/CUDA wheel indirectly through dependency resolution.
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--upgrade",
            "--no-deps",
            *repair_requirements,
        ],
        check=True,
    )
    importlib.invalidate_caches()
    runtime_status = {
        package_name: runtime_package_status(package_name)
        for package_name in RUNTIME_IMPORTS
    }

invalid_packages = {
    package_name: {
        "status": status,
        "installed_version": installed_version,
        "required": str(requirement),
    }
    for package_name, (status, installed_version, requirement) in runtime_status.items()
    if status != "ok"
}
RUNTIME_REQUIREMENTS_VALID = not invalid_packages
if invalid_packages and STRICT_RUNTIME_REQUIREMENTS:
    raise ImportError(
        "Runtime packages do not satisfy requirements.txt: "
        + json.dumps(invalid_packages, sort_keys=True)
    )
RUNTIME_PACKAGE_VERSIONS = {
    declared_requirements[package_name].name: installed_version
    for package_name, (_, installed_version, _) in runtime_status.items()
}
if invalid_packages and not STRICT_RUNTIME_REQUIREMENTS:
    print(
        "Engineering fixture warning: runtime packages are outside the declared "
        "ranges. This run is claim-ineligible by construction: "
        + json.dumps(invalid_packages, sort_keys=True)
    )

try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True
    ).strip()
except (OSError, subprocess.CalledProcessError):
    GIT_COMMIT = None

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)
print({
    "project_root": str(PROJECT_ROOT),
    "git_commit": GIT_COMMIT,
    "quick_run": QUICK_RUN,
    "test_fixture": USE_TEST_FIXTURE,
    "claim_eligible_mode": not QUICK_RUN and not USE_TEST_FIXTURE,
    "strict_runtime_requirements": STRICT_RUNTIME_REQUIREMENTS,
    "runtime_requirements_valid": RUNTIME_REQUIREMENTS_VALID,
    "runtime_package_versions": RUNTIME_PACKAGE_VERSIONS,
})

## 1. Chế độ chạy và accelerator

Full mode là mặc định. Trên Kaggle nên chọn GPU T4. Bộ giải quyết device sẽ kiểm tra CUDA
compute capability; nếu wheel không tương thích (thường gặp với P100/sm_60), neural model
chuyển về CPU và ghi rõ fallback trong provenance thay vì phát sinh lỗi kernel khó chẩn đoán.

In [ ]:
from src.stress_testing.artifacts import load_yaml_config
from src.stress_testing.predictors import resolve_safe_torch_device

CONFIG_PATH = PROJECT_ROOT / 'configs/stress/transxion_v2.yaml'
CONFIG = load_yaml_config(CONFIG_PATH)
OUTPUT_DIR = OUTPUT_BASE / '09_transxion_v2_stress_test'
DEVICE = resolve_safe_torch_device("cuda")
print({
    "config": str(CONFIG_PATH),
    "output_dir": str(OUTPUT_DIR),
    "requested_device": DEVICE.requested,
    "selected_device": DEVICE.selected,
    "device_reason": DEVICE.reason,
    "dataset_role": CONFIG["project"]["claim_scope"],
    "coverage_budgets": CONFIG["evaluation"]["coverages"],
})

## 2. Input discovery, identity and checksum preflight

Full run phải tìm đúng một file có tên chính xác trong các input roots. Loader sẽ từ chối
Git-LFS pointer, nhiều file trùng tên, schema sai, target không nhị phân và checksum sai.
Checksum giúp định danh release; filename hoặc số dòng riêng lẻ không đủ làm provenance.

In [ ]:
from src.stress_testing.data import (
    make_synthetic_transxion_test_fixture, file_checksum, resolve_stress_data_file
)

if USE_TEST_FIXTURE:
    # Use the exact directory/row contract consumed by
    # run_stress_test so preflight and execution identify one file.
    FIXTURE_ROOT = OUTPUT_DIR / ".stress_test_fixture"
    fixture_rows = int(CONFIG.get("testing", {}).get("fixture_rows", 800))
    fixture_result = make_synthetic_transxion_test_fixture(
        FIXTURE_ROOT,
        n_rows=fixture_rows,
        seed=int(CONFIG["project"].get("seed", 42)),
    )
    DATA_ROOTS = [FIXTURE_ROOT]

source_path = resolve_stress_data_file('tx.csv', DATA_ROOTS)
checksum_config = CONFIG["dataset"]["checksum"]
observed_checksum = file_checksum(source_path, checksum_config["algorithm"])
preflight = {
    "source_path": str(source_path),
    "bytes": source_path.stat().st_size,
    "checksum_algorithm": checksum_config["algorithm"],
    "observed_checksum": observed_checksum,
    "expected_checksum": checksum_config["value"],
    "checksum_match": observed_checksum.lower() == checksum_config["value"].lower(),
    "verification_required": not USE_TEST_FIXTURE,
}
display(preflight)
if not USE_TEST_FIXTURE and not preflight["checksum_match"]:
    raise ValueError("The attached input is not the proposal-pinned dataset release")

## 3. Execute the locked stress-test state machine

`run_stress_test` thực hiện tuần tự các bước sau:

1. Chuẩn hóa schema, dựng đặc trưng lịch sử chỉ từ các giao dịch quá khứ và kiểm tra leakage denylist.
2. Split thời gian 60/20/20; validation được chia tiếp cho calibration, model selection, rule audit và policy selection.
3. Train XGBoost, LightGBM và TabularResNetV2 với xử lý class imbalance.
4. Chọn calibrator, F2 threshold và reference predictor chỉ bằng validation rồi đóng băng chúng.
5. Fit Tier A/B candidates trên train; giữ Tier C làm baseline tách biệt; audit attribution, TP-FP, lift, stability và redundancy chỉ trong frozen-predictor alert region của validation.
6. Fit contrastive meta-scorer từ audited rule truth và frozen calibrated risk chỉ trên TP/FP alerts của `rule_audit`; không fit lại trên `policy_select` hoặc test.
7. Chọn individual method hoặc pre-registered guarded ensemble, hoặc abstain, ở từng coverage bằng score-only và stability guardrails.
8. Sau khi policy đã khóa mới tính test metrics, matched-risk, residual TP-FP và paired block bootstrap.
9. Chạy negative controls; phân loại kết quả dương, âm, abstention hoặc saturation; ghi checksum/lineage.

In [ ]:
from src.stress_testing.experiment import run_stress_test

run_result = run_stress_test(
    CONFIG_PATH,
    data_roots=DATA_ROOTS,
    output_dir=OUTPUT_DIR,
    quick_run=QUICK_RUN,
    use_test_fixture=USE_TEST_FIXTURE,
    device=DEVICE.selected,
)
print(run_result)

## 4. Data quality, provenance and temporal integrity

In [ ]:
def read_json(name):
    return json.loads((OUTPUT_DIR / name).read_text(encoding="utf-8"))

data_manifest = read_json("data_manifest.json")
split_integrity = pd.read_csv(OUTPUT_DIR / "split_integrity.csv")
display(pd.DataFrame(data_manifest["source_files"]))
display(pd.DataFrame(data_manifest["data_quality"].items(), columns=["check", "value"]))
display(split_integrity)

assert data_manifest["data_quality"]["timestamp_parse_coverage"] == 1.0
assert data_manifest["data_quality"]["target_missing_count"] == 0
assert split_integrity["both_classes"].all()
assert split_integrity["chronologically_disjoint"].all()

## 5. Predictive benchmark and frozen reference predictor

PR-AUC là metric chọn predictor chính do class imbalance. ROC-AUC, F2, precision, recall,
Recall@1% FPR, Brier, ECE và NLL được báo cùng prevalence/null baseline. Việc model có metric
cao không tự động chứng minh explanation có incremental value.

In [ ]:
predictive = pd.read_csv(OUTPUT_DIR / "predictive_metrics.csv")
calibration = pd.read_csv(OUTPUT_DIR / "calibration_comparison.csv")
predictor_manifest = read_json("predictor_manifest.json")
display(predictive.sort_values(["selection_split_pr_auc", "family"], ascending=[False, True]))
display(calibration)
display({key: predictor_manifest.get(key) for key in [
    "reference_family", "reference_seed", "threshold", "calibration_method", "frozen_before_explanation"
]})
assert predictor_manifest["frozen_before_explanation"] is True

## 6. Candidate registry, validation-only audit and deduplication

In [ ]:
registry = pd.read_csv(OUTPUT_DIR / "rule_registry.csv")
audit = pd.read_csv(OUTPUT_DIR / "rule_audit.csv")
redundancy = pd.read_csv(OUTPUT_DIR / "rule_redundancy.csv")
counterfactual = pd.read_csv(OUTPUT_DIR / "counterfactual_diagnostics.csv")
candidate_provenance = read_json("candidate_generation_provenance.json")
surrogate_rules = pd.read_csv(OUTPUT_DIR / "surrogate_rules.csv")
surrogate_provenance = read_json("surrogate_provenance.json")
ablation_validation = pd.read_csv(OUTPUT_DIR / "ablation_validation_results.csv")
ablation_stability = pd.read_csv(OUTPUT_DIR / "ablation_validation_stability.csv")
ablation_rule_selection = pd.read_csv(OUTPUT_DIR / "ablation_rule_selection.csv")
ablation_rule_pool_provenance = read_json("ablation_rule_pool_provenance.json")
ablation = pd.read_csv(OUTPUT_DIR / "ablation_results.csv")
predictor_sensitivity = pd.read_csv(
    OUTPUT_DIR / "predictor_explanation_sensitivity.csv"
)
predictor_attribution = pd.read_csv(
    OUTPUT_DIR / "predictor_attribution_sensitivity.csv"
)
contrastive_coefficients = pd.read_csv(
    OUTPUT_DIR / "contrastive_meta_coefficients.csv"
)
contrastive_provenance = read_json("contrastive_meta_provenance.json")
predictor_contrastive = pd.read_csv(
    OUTPUT_DIR / "predictor_contrastive_sensitivity.csv"
)
predictor_contrastive_provenance = read_json(
    "predictor_contrastive_sensitivity_provenance.json"
)
display(registry.groupby(["tier", "source"], dropna=False).size().rename("candidate_count"))
display(counterfactual)
display(candidate_provenance)
display(surrogate_rules)
display(surrogate_provenance)
display(audit.sort_values(["selected", "audit_alert_score"], ascending=[False, False]))
display(redundancy.head(30))
display(ablation_validation[ablation_validation["coverage"].round(2).isin([0.10, 0.25])])
display(ablation_stability[ablation_stability["coverage"].round(2).isin([0.10, 0.25])])
display(ablation_rule_selection)
display(ablation_rule_pool_provenance)
display(ablation[ablation["coverage_budget"].round(2).isin([0.10, 0.25])])
display(predictor_sensitivity[
    predictor_sensitivity["coverage_budget"].round(2).isin([0.10, 0.25])
])
display(predictor_attribution.head(30))
display(contrastive_provenance)
display(contrastive_coefficients)
display(predictor_contrastive.head(30))
display(predictor_contrastive_provenance)

## 7. Locked selective policies and fixed-coverage results

Một coverage result chỉ được diễn giải là lợi ích bổ sung nếu nó vượt predictor-score-only
ở đúng cùng số lượng alert thực tế được chọn. Nếu rule support trên test không đủ requested
budget, VASRE được phép underfill và comparator chính cũng bị khóa về realized count; bảng vẫn
giữ requested-budget score-only như diagnostic riêng. Nếu validation guardrail không đạt, hệ
thống phải abstain; đây là kết quả hợp lệ, không phải lỗi pipeline.

In [ ]:
policies = pd.DataFrame(read_json("locked_policies.json"))
candidates = pd.read_csv(OUTPUT_DIR / "policy_candidates_validation.csv")
coverage = pd.read_csv(OUTPUT_DIR / "coverage_results.csv")
display(policies)
display(coverage)

plot_frame = coverage.copy()
fig, ax = plt.subplots(figsize=(11, 6), constrained_layout=True)
ax.plot(plot_frame["coverage_budget"] * 100, plot_frame["selected_alert_precision"], marker="o", label="VASRE")
ax.plot(plot_frame["coverage_budget"] * 100, plot_frame["score_only_precision"], marker="s", label="Predictor score only")
ax.axhline(plot_frame["all_alert_precision"].iloc[0], color="grey", linestyle=":", label="All alerts")
ax.set(xlabel="Explanation coverage among alerts (%)", ylabel="Fraud precision", title="Fixed-budget alert triage")
ax.legend(loc="best")
plt.show()

## 8. Matched-risk, residual evidence and paired uncertainty

In [ ]:
matched = pd.read_csv(OUTPUT_DIR / "matched_risk_results.csv")
residual = pd.read_csv(OUTPUT_DIR / "residual_evidence.csv")
bootstrap = pd.read_csv(OUTPUT_DIR / "paired_bootstrap.csv")
outcomes = pd.read_csv(OUTPUT_DIR / "stress_outcomes.csv")
display(matched)
display(residual)
display(bootstrap)
display(outcomes)

primary = coverage[coverage["coverage_budget"].round(2).isin([0.10, 0.25])]
if len(primary):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    sns.barplot(data=primary, x="coverage_budget", y="delta_vs_score_only", ax=axes[0], color="#4c72b0")
    axes[0].axhline(0, color="black", linewidth=1)
    axes[0].set(title="Incremental precision vs score only", xlabel="Coverage", ylabel="Precision difference")
    primary_outcomes = outcomes[outcomes["coverage_budget"].round(2).isin([0.10, 0.25])]
    sns.barplot(data=primary_outcomes, x="coverage_budget", y="risk_conditioned_tp_fp_gap", ax=axes[1], color="#55a868")
    axes[1].axhline(0, color="black", linewidth=1)
    axes[1].set(title="Risk-conditioned residual TP-FP evidence", xlabel="Coverage", ylabel="Evidence gap")
    plt.show()

## 9. Stability, negative controls, saturation and claim boundary

In [ ]:
stability = pd.read_csv(OUTPUT_DIR / "validation_stability.csv")
rule_stability = pd.read_csv(OUTPUT_DIR / "rule_audit_stability.csv")
controls = pd.read_csv(OUTPUT_DIR / "negative_controls.csv")
summary = read_json("stress_summary.json")
manifest = read_json("stress_test_manifest.json")
lineage = read_json("stress_lineage.json")
display(stability)
display(rule_stability)
display(controls)
display(summary)
display({
    "claim_eligible": manifest["claim_eligible"],
    "claim_blockers": manifest["claim_blockers"],
    "stress_pipeline_fingerprint": manifest["stress_pipeline_fingerprint"],
    "lineage_outputs": len(lineage["outputs"]),
})

if QUICK_RUN or USE_TEST_FIXTURE:
    assert manifest["claim_eligible"] is False

## 10. Diễn giải kết quả đúng phạm vi

- Kết quả dương chỉ được xác nhận khi paired bootstrap, matched-risk, residual evidence,
  validation stability và negative controls cùng hỗ trợ; label-shuffle và weight-shuffle
  validation controls là hai cổng bắt buộc. Một CI dương riêng lẻ là chưa đủ.
- Domain Tier A only, exact counterfactual-only và all Tier B tự deduplicate/select/weight từ
  complete pre-dedup audit pool; chúng không chỉ lọc các rule đã thắng trong primary A/B pool.
- Kết quả âm hoặc abstention cho thấy rule evidence chưa tạo incremental value trong protocol này.
- Rule audit, TP-FP metrics, weighting và deduplication chỉ dùng frozen-predictor alerts; TN/FN ngoài alert region chỉ xuất hiện trong population diagnostics.
- Counterfactual-only là đúng tập rule có source `train_model_counterfactual`; `train_derived_tier_b_only` rộng hơn và được báo riêng.
- Tier C CART và signed native-attribution top-k là baseline chẩn đoán, không được nhập vào primary A/B policy.
- Contrastive meta-scorer dùng audited rule truth cùng frozen calibrated risk và chỉ fit trên TP/FP alerts của `rule_audit`; fallback thiếu mẫu/lớp được báo rõ và không được âm thầm fit trên test.
- Guarded ensemble là trung bình của các component đã khóa trong YAML, sau đó vẫn phải qua score-only và resampling-stability guardrails trên `policy_select`.
- Saturation chỉ được xác nhận cho AMLNet full/checksummed run khi đủ positive, matched pairs,
  bootstrap replicates và toàn bộ lineage/attribution guardrails; TransXion không mang nhãn này.
- Matched-risk và residual TP-FP làm giảm một số confounding theo risk score nhưng không chứng minh nhân quả.
- Stress test không chứng minh production generalization hoặc hiệu quả với chuyên viên AML thực tế.

Hãy lưu một Kaggle version hoàn tất để bảo toàn toàn bộ files trong `OUTPUT_DIR`. Notebook 11
chỉ tổng hợp version outputs này và không được chọn lại model, rule hoặc policy.